# **Feature Engineering**

## Objectives

- Prepare the cleaned Premier League player dataset for machine learning.
- Select appropriate predictor features based on findings from exploratory data analysis.
- Remove features that would cause target leakage.
- Investigate and handle missing values required for modelling.
- Transform categorical variables into a suitable numerical representation.
- Prepare and save the engineered dataset for model development.

## Inputs

- `data/processed/all_players_cleaned.csv` – Cleaned player-season dataset produced by the Data Cleaning notebook.

## Outputs

- A feature-engineered dataset suitable for machine learning and model development.

## Additional Comments

- The `HighScorer` target represents players who scored 10 or more Premier League goals in a season.
- EDA identified substantial class imbalance, correlated attacking statistics and systematic missing values in several shooting features. These findings will be considered when preparing the features for modelling.

### Imports

In [1]:
import os
import pandas as pd
import numpy as np

### Change Working directory

In [4]:
current_dir = os.getcwd()
current_dir

'c:\\code\\premier-league-predictor\\premier-league-predictor\\jupyter_notebooks'

In [5]:
os.chdir(r"C:\code\premier-league-predictor\premier-league-predictor")
print("You set a new current directory")

You set a new current directory


In [6]:
current_dir = os.getcwd()
current_dir

'C:\\code\\premier-league-predictor\\premier-league-predictor'

In [7]:
df = pd.read_csv("data/processed/all_players_cleaned.csv")

In [8]:
df.shape

(8196, 54)

In [9]:
df.head()

,Name,Position,Appearances,Clean sheets,Goals conceded,Tackles,Tackle success %,Last man tackles,Blocked shots,Interceptions,...,Big chances missed,Saves,Penalties saved,Punches,High Claims,Catches,Sweeper clearances,Throw outs,Goal Kicks,Season
0,Rolando Aarons,Midfielder,10,NaN,NaN,13.0,77.0,NaN,0.0,6.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2015-16
1,Almen Abdi,Midfielder,32,NaN,NaN,83.0,78.0,NaN,10.0,32.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2015-16
2,Abdul Rahman Baba,Defender,15,2.0,13.0,47.0,83.0,0.0,1.0,23.0,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2015-16
3,Mehdi Abeid,Midfielder,0,NaN,NaN,0.0,0.0,NaN,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2015-16
4,Tammy Abraham,Forward,2,NaN,NaN,0.0,NaN,NaN,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2015-16


## Define Target Variable

The machine learning model will predict whether a player is classified as a `HighScorer`, defined as scoring 10 or more Premier League goals in a season.

The target variable is recreated from `Goals` before goal-derived features are removed from the predictor dataset.

In [10]:
df.columns

Index(['Name', 'Position', 'Appearances', 'Clean sheets', 'Goals conceded',
       'Tackles', 'Tackle success %', 'Last man tackles', 'Blocked shots',
       'Interceptions', 'Clearances', 'Headed Clearance',
       'Clearances off line', 'Recoveries', 'Duels won', 'Duels lost',
       'Successful 50/50s', 'Aerial battles won', 'Aerial battles lost',
       'Own goals', 'Errors leading to goal', 'Assists', 'Passes',
       'Passes per match', 'Big chances created', 'Crosses',
       'Cross accuracy %', 'Through balls', 'Accurate long balls',
       'Yellow cards', 'Red cards', 'Fouls', 'Offsides', 'Goals',
       'Headed goals', 'Goals with right foot', 'Goals with left foot',
       'Hit woodwork', 'Goals per match', 'Penalties scored',
       'Freekicks scored', 'Shots', 'Shots on target', 'Shooting accuracy %',
       'Big chances missed', 'Saves', 'Penalties saved', 'Punches',
       'High Claims', 'Catches', 'Sweeper clearances', 'Throw outs',
       'Goal Kicks', 'Season'],
     

In [11]:
df["HighScorer"] = df["Goals"] >= 10

In [12]:
df["HighScorer"].value_counts()

HighScorer
False    7984
True      212
Name: count, dtype: int64

### Remove Target Leakage Features

The `HighScorer` target is derived from the number of goals scored by each player. Therefore, `Goals` cannot be used as a predictor because it would directly reveal information used to determine the target.

Other statistics that directly describe how those goals were scored are also excluded to reduce target leakage. The model should instead learn from player characteristics and performance statistics that do not directly provide the outcome being predicted.

In [14]:
goal_columns = [
    "Goals",
    "Headed goals",
    "Goals with right foot",
    "Goals with left foot",
    "Goals per match",
    "Penalties scored",
    "Freekicks scored"
]

In [15]:
goal_columns

['Goals',
 'Headed goals',
 'Goals with right foot',
 'Goals with left foot',
 'Goals per match',
 'Penalties scored',
 'Freekicks scored']

In [21]:
x = df.drop(columns=goal_columns + ["HighScorer"])
y = df["HighScorer"]

In [22]:
x.shape

(8196, 47)

In [23]:
y.shape

(8196,)

In [24]:
x.columns

Index(['Name', 'Position', 'Appearances', 'Clean sheets', 'Goals conceded',
       'Tackles', 'Tackle success %', 'Last man tackles', 'Blocked shots',
       'Interceptions', 'Clearances', 'Headed Clearance',
       'Clearances off line', 'Recoveries', 'Duels won', 'Duels lost',
       'Successful 50/50s', 'Aerial battles won', 'Aerial battles lost',
       'Own goals', 'Errors leading to goal', 'Assists', 'Passes',
       'Passes per match', 'Big chances created', 'Crosses',
       'Cross accuracy %', 'Through balls', 'Accurate long balls',
       'Yellow cards', 'Red cards', 'Fouls', 'Offsides', 'Hit woodwork',
       'Shots', 'Shots on target', 'Shooting accuracy %', 'Big chances missed',
       'Saves', 'Penalties saved', 'Punches', 'High Claims', 'Catches',
       'Sweeper clearances', 'Throw outs', 'Goal Kicks', 'Season'],
      dtype='object')

### Target and Leakage Observations

The `HighScorer` target contains **212 high-scorer records** and **7,984 non-high-scorer records** across 8,196 player-season observations.

Seven goal-derived features were excluded from the predictor dataset to prevent target leakage. The target variable was also separated from the predictors, resulting in **47 candidate predictor features**.

Further feature selection is required before modelling, as not all remaining features are expected to provide useful information for predicting high scorers.